# Agent 1 — ทดลองคุยกับ Gemini จริง

ใช้ `Agent1.handle()` ตัวเดียวกับ API รวม Privacy Filter, Extractor, Priority, Knowledge Base และ Response Generator จริง แต่ไม่ได้ผ่าน HTTP หรือ LINE และไม่ส่งข้อความหาใคร

ข้อมูลเก็บใน SQLite ในหน่วยความจำ ใช้ข้อมูลสมมติสำหรับทดลอง โมเดลเรียก Gemini จริงและอาจมีค่าใช้จ่ายตามบัญชี

1. เลือก Python kernel แล้วรันเซลล์ติดตั้งเมื่อยังไม่มี dependencies
2. ใส่ `GEMINI_API_KEY` หรือ `GOOGLE_API_KEY` ใน `lab/1/.env` (หรือ `dspy-agent/.env`)
3. รันเซลล์ตั้งค่าและฟังก์ชัน จากนั้นเริ่มคุยในเซลล์โหมดต่อเนื่อง


In [1]:
# รันครั้งแรก หากเพิ่งติดตั้งให้ restart kernel แล้วรันเซลล์ถัดไป
from pathlib import Path

BASE = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "dspy-agent" / "app" / "agent.py").is_file()), None)
if BASE is None:
    raise RuntimeError("กรุณาเปิด notebook ภายในโฟลเดอร์ DSI490")
%pip install -r {str(BASE / "dspy-agent" / "requirements-notebook.txt")}


Note: you may need to restart the kernel to use updated packages.


## Mock chat — หน้าจอแชตจำลอง

รันเซลล์ติดตั้งด้านบน แล้วรันเซลล์นี้ได้เลย **โหมด Mock ไม่ต้องตั้ง API key** คำตอบเป็นบทสนทนาตัวอย่างตามลำดับ ไม่ได้ประมวลผลข้อความหรือบันทึกเคสจริง

เลือก **Agent จริง** หลังรันเซลล์ตั้งค่าและเซลล์ที่ประกาศ `reset()` แล้ว เพื่อคุยกับ Gemini ผ่านหน้าจอนี้ (มีการเรียก API จริง) ใช้ข้อมูลสมมติเท่านั้น

ในโหมด Mock ลองส่ง “สวัสดี” → กดยินยอม → พิมพ์แจ้งเหตุ → ส่งหมุด → พิมพ์ชื่อ → พิมพ์เบอร์ → กดยืนยัน ตัวเลขและข้อความสรุปเป็นตัวอย่างคงที่


In [2]:
import html
import ipywidgets as widgets
from IPython.display import display

chat_mode = widgets.Dropdown(options=["Mock", "Agent จริง"], description="โหมด:")
chat_log = widgets.HTML()
chat_text = widgets.Text(placeholder="พิมพ์ข้อความสมมติ…", layout=widgets.Layout(width="75%"))
chat_send = widgets.Button(description="ส่ง", button_style="primary")
chat_reset = widgets.Button(description="เริ่มแชตใหม่")
chat_lat = widgets.FloatText(value=14.9512, description="Latitude:")
chat_lng = widgets.FloatText(value=100.3646, description="Longitude:")
chat_pin = widgets.Button(description="ส่งหมุด")
chat_buttons = widgets.HBox()
chat_status = widgets.Label()
chat_history = []
mock_step = 0

def paint_chat():
    bubbles = []
    for who, text in chat_history:
        align, color = ("right", "#d9fdd3") if who == "คุณ" else ("left", "#ffffff")
        safe = html.escape(text).replace("\n", "<br>")
        bubbles.append(f'<div style="text-align:{align};margin:12px 0"><div style="display:inline-block;max-width:85%;text-align:left;background:{color};color:#17212b;padding:12px 16px;border-radius:16px"><b>{who}</b><br>{safe}</div></div>')
    chat_log.value = '<div style="background:#edf1f5;padding:16px;border-radius:16px;max-height:500px;overflow:auto">' + ("".join(bubbles) or "เริ่มคุยโดยพิมพ์ข้อความด้านล่าง") + '</div>'

def set_chat_buttons(items):
    buttons = []
    for label, data in items:
        button = widgets.Button(description=label)
        button.on_click(lambda _, label=label, data=data: chat_turn("postback", data, label))
        buttons.append(button)
    chat_buttons.children = tuple(buttons)

def mock_answer(kind, value):
    global mock_step
    if mock_step == 0:
        if kind == "postback" and value == "consent=yes":
            mock_step = 1
            return "[ตัวอย่าง] ขอบคุณครับ กรุณาแจ้งปัญหา จำนวนผู้ประสบภัย และความช่วยเหลือที่ต้องการ", []
        return "[ตัวอย่าง] ระบบขอความยินยอมเก็บข้อมูลเพื่อรับแจ้งเหตุครับ", [("ยินยอม", "consent=yes")]
    if mock_step == 1:
        mock_step = 2
        return "[ตัวอย่าง] รับข้อมูลแล้วครับ กรุณาส่งหมุดจุดเกิดเหตุด้วยช่องพิกัดด้านล่าง", []
    if mock_step == 2:
        if kind != "location":
            return "[ตัวอย่าง] กรุณากดส่งหมุดด้านล่างก่อนครับ", []
        mock_step = 3
        return "[ตัวอย่าง] ได้รับหมุดแล้วครับ ขอชื่อและนามสกุลสมมติสำหรับติดต่อครับ", []
    if mock_step == 3:
        mock_step = 4
        return "[ตัวอย่าง] ขอเบอร์โทรสมมติสำหรับติดต่อครับ", []
    if mock_step == 4:
        mock_step = 5
        return "[ตัวอย่างคงที่] สรุปข้อมูล\nปัญหา: น้ำท่วมบ้าน\nผู้ประสบภัย: ผู้ใหญ่ 2 คน\nต้องการ: เรือ\nจุดเกิดเหตุ: ได้รับหมุดแล้ว\nกรุณาตรวจสอบและยืนยันข้อมูล", [("ยืนยันข้อมูล", "mock_confirm")]
    if kind == "postback" and value == "mock_confirm":
        return "[ตัวอย่าง] ยืนยันข้อมูลแล้วครับ (โหมด Mock ไม่ได้สร้างเคสจริง)", []
    return "จบบทสนทนาตัวอย่างแล้ว กดเริ่มแชตใหม่เพื่อลองอีกครั้งครับ", []

def chat_turn(kind, value, label=None):
    chat_send.disabled = chat_pin.disabled = True
    for button in chat_buttons.children:
        button.disabled = True
    chat_status.value = "กำลังประมวลผล…"
    chat_history.append(("คุณ", label or (f"ส่งหมุด {value}" if kind == "location" else str(value))))
    paint_chat()
    try:
        if chat_mode.value == "Mock":
            text, buttons = mock_answer(kind, value)
        else:
            if globals().get("agent") is None:
                raise RuntimeError("รันเซลล์ตั้งค่า Gemini และเซลล์ประกาศ reset() ก่อนเลือก Agent จริง")
            kwargs = {"text" if kind == "text" else kind: value}
            out = agent.handle(USER, **kwargs)
            globals()["last_output"] = out
            text = out.output_text
            buttons = [(q.label, q.data) for q in out.quick_replies if q.type != "location"]
        chat_history.append(("Agent", text))
        set_chat_buttons(buttons)
        chat_status.value = "โหมด: " + chat_mode.value
    except Exception as exc:
        chat_status.value = "ทำรายการไม่สำเร็จ: " + type(exc).__name__ + (" — รันเซลล์ตั้งค่าก่อน" if globals().get("agent") is None and chat_mode.value != "Mock" else " — ตรวจการตั้งค่าและลองใหม่")
    finally:
        chat_send.disabled = chat_pin.disabled = False
        for button in chat_buttons.children:
            button.disabled = False
        paint_chat()

def send_chat(_):
    message = chat_text.value.strip()
    if message:
        chat_text.value = ""
        chat_turn("text", message)

def restart_chat(_=None):
    global mock_step
    if chat_mode.value == "Agent จริง":
        if not callable(globals().get("reset")):
            chat_status.value = "รันเซลล์ตั้งค่าและเซลล์ประกาศ reset() ก่อน"
            return
        reset()
    mock_step = 0
    chat_history.clear()
    set_chat_buttons([])
    chat_status.value = "โหมด: " + chat_mode.value
    paint_chat()

def send_chat_pin(_):
    if not (-90 <= chat_lat.value <= 90 and -180 <= chat_lng.value <= 180):
        chat_status.value = "พิกัดอยู่นอกช่วงที่อนุญาต"
        return
    chat_turn("location", (chat_lat.value, chat_lng.value))

chat_send.on_click(send_chat)
chat_pin.on_click(send_chat_pin)
chat_reset.on_click(restart_chat)
chat_mode.observe(restart_chat, names="value")
paint_chat()
display(widgets.VBox([widgets.HBox([chat_mode, chat_reset]), chat_log,
                      chat_buttons, widgets.HBox([chat_text, chat_send]),
                      widgets.HBox([chat_lat, chat_lng, chat_pin]), chat_status]))


In [3]:
import os
import sys
from pathlib import Path

BASE = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "dspy-agent" / "app" / "agent.py").is_file()), None)
if BASE is None:
    raise RuntimeError("กรุณาเปิด notebook ภายในโฟลเดอร์ DSI490")
ROOT = BASE / "dspy-agent"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import dspy
from dotenv import load_dotenv

for env_path in (BASE / "lab" / "1" / ".env", ROOT / ".env"):
    load_dotenv(env_path, override=False)
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise RuntimeError("ยังไม่มี API key: เพิ่ม GEMINI_API_KEY ใน lab/1/.env แล้วรันเซลล์นี้ใหม่")
MODEL = os.getenv("GEMINI_MODEL", "gemini/gemini-3.1-flash-lite")
dspy.configure(lm=dspy.LM(model=MODEL, api_key=api_key))
print("พร้อมใช้โมเดล:", MODEL)


พร้อมใช้โมเดล: gemini/gemini-3.1-flash-lite


In [4]:
from app.agent import Agent1
from app.knowledge import KnowledgeBase
from app.store import Store

USER = "lab_user"
last_output = None
agent = None


def reset():
    global agent, last_output
    if agent is not None:
        agent.store.close()
    kb = KnowledgeBase(ROOT / "app" / "knowledge_th.csv", allow_unreviewed=True)
    agent = Agent1(Store("sqlite:///:memory:"), kb)
    last_output = None
    print("เริ่มใหม่แล้ว: ล้างข้อมูลทดลองทุกเคส / ใช้ KB mock สำหรับทดลอง")


def show(out):
    global last_output
    last_output = out
    print("\nAgent:", out.output_text)
    for i, q in enumerate(out.quick_replies, 1):
        print(f"  [{i}] {q.label}" + (" (ส่งหมุดด้วย /pin)" if q.type == "location" else ""))
    print(f"\nเคส: {out.case_id} | สถานะ: {out.intake_status.value if out.intake_status else '-'}"
          f" | รอตอบ: {out.pending_question} | KB: {out.kb_doc_ids}")
    return out


def say(text):
    return show(agent.handle(USER, text=text))


def pin(lat=14.9512, lng=100.3646):
    return show(agent.handle(USER, location=(lat, lng)))


def press(button):
    if isinstance(button, int):
        if last_output is None or not 1 <= button <= len(last_output.quick_replies):
            raise ValueError("เลือกหมายเลขปุ่มจากคำตอบล่าสุด")
        q = last_output.quick_replies[button - 1]
        if q.type == "location":
            raise ValueError("ปุ่มนี้ต้องส่งหมุด: ใช้ pin(lat, lng) หรือ /pin lat lng")
        button = q.data
    return show(agent.handle(USER, postback=button))


def inspect_case():
    case_id = agent.store.user(USER)["current_case_id"]
    case = agent.store.load_case(case_id) if case_id else None
    if case is None:
        print("ยังไม่มีเคส")
        return
    print("Incident:", case.incident.model_dump())
    print("Priority:", case.priority.model_dump(), case.priority_rule_id)
    print("Risk facts:", {k: v for k, v in case.risk_facts.model_dump().items() if v != "unknown"})
    print("Slots:", {k.value: (v.status.value, v.ask_count) for k, v in case.slots.items()})
    print("Draft revision:", case.draft_rev, "| Confirmed version:", case.current_version)

reset()


เริ่มใหม่แล้ว: ล้างข้อมูลทดลองทุกเคส / ใช้ KB mock สำหรับทดลอง


## คุยต่อเนื่อง

รันเซลล์ด้านล่าง แล้วพิมพ์ในช่อง input ของ notebook ทีละข้อความ ระบบจำเคสให้เอง

- `/press 1` — กดปุ่มหมายเลข 1 จากคำตอบล่าสุด รวมถึงปุ่ม consent และยืนยันข้อมูล
- `/pin 14.9512 100.3646` — ส่งตำแหน่งสมมติ
- `/state` — ดูข้อมูลเคสและช่องที่ยังขาด
- `/pipeline` — ดูข้อมูลที่ยืนยันแล้ว
- `/reset` — ล้างข้อมูลทดลองทุกเคส
- `/exit` — ออกจากลูป โดยยังเก็บเคสไว้ คุยต่อด้วยฟังก์ชันหรือรันลูปใหม่ได้

ลองเริ่มด้วย “สวัสดีครับ” แล้วกดปุ่มยินยอม ก่อนแจ้ง “น้ำท่วมบ้านสูงถึงเอว ออกไม่ได้ มีผู้ใหญ่ 2 คน ต้องการเรือ” จากนั้นตอบตามที่ Agent ถามจนถึงปุ่มยืนยัน

โหมดนี้ไม่มี background job เตือนอัตโนมัติแบบ API และใช้ KB mock ที่ยังไม่ผ่านการตรวจรับรอง


In [ ]:
while True:
    message = input("คุณ: ").strip()
    if not message:
        continue
    if message == "/exit":
        break
    try:
        if message == "/reset":
            reset()
        elif message == "/state":
            inspect_case()
        elif message == "/pipeline":
            print(agent.store.pipeline_rows())
        elif message.startswith("/press "):
            press(int(message.split()[1]))
        elif message.startswith("/pin "):
            _, lat, lng = message.split()
            lat, lng = float(lat), float(lng)
            if not (-90 <= lat <= 90 and -180 <= lng <= 180):
                raise ValueError("พิกัดอยู่นอกช่วงที่อนุญาต")
            pin(lat, lng)
        elif message.startswith("/"):
            print("คำสั่ง: /press N, /pin lat lng, /state, /pipeline, /reset, /exit")
        else:
            say(message)
    except Exception as exc:
        print("ทำรายการไม่สำเร็จ:", type(exc).__name__)
        print("ตรวจคำสั่ง API key โมเดล และการเชื่อมต่อ แล้วลองใหม่ ข้อมูลก่อนหน้าอาจถูกบันทึกแล้ว")


## หรือคุยทีละเซลล์

ออกจากลูปด้วย `/exit` ก่อน จากนั้นแก้คำสั่งด้านล่างแล้วรันซ้ำได้ เช่น `say("ข้อความ")`, `press(1)`, `pin()` หรือ `inspect_case()`

หากจะเริ่มบทสนทนาใหม่ทั้งหมด ใช้ `reset()` ก่อน


In [ ]:
# ตัวอย่าง: เอา # ออกเฉพาะคำสั่งที่ต้องการลอง
# say("สวัสดีครับ")
# press(1)  # ตรวจข้อความบนปุ่มล่าสุดก่อนเลือก
# say("น้ำท่วมบ้านสูงถึงเอว ออกไม่ได้ มีผู้ใหญ่ 2 คน ต้องการเรือ")
# pin()
# inspect_case()


In [ ]:
# จะเห็นข้อมูลเฉพาะเคสที่กดยืนยันแล้ว
agent.store.pipeline_rows()


## ดูข้อมูลที่บันทึกลง Database จริง

รันเซลล์นี้หลังสร้าง Agent แล้ว กด **โหลดข้อมูลที่บันทึก** หลังคุยแต่ละรอบ เลือก user_id ให้ตรงกับหน้าจอแชต

เปิด `case_personal_data` เพื่อดูชื่อ นามสกุล เบอร์ และพิกัด; `case_drafts` เพื่อดูร่าง; หลังยืนยันดู `case_versions` และ `case_version_contacts` รวมถึง View ของ Pipeline

ส่วนนี้อ่านแถวจากฐานข้อมูลโดยตรง ไม่ใช่ข้อความตอบของ LLM โหมด Mock ไม่สร้างแถวจริง SQLite ในหน่วยความจำจะหายเมื่อ reset หรือปิด kernel และข้อมูลส่วนบุคคลอาจติดใน output เมื่อบันทึก notebook จึงควรใช้ข้อมูลสมมติ


In [ ]:
import json
import ipywidgets as widgets
from IPython.display import display

# อ่านจาก connection ของ Agent โดยตรง ไม่ต้องใช้ admin token หรือ HTTP
saved_user = widgets.Text(
    value=globals().get("current_user", globals().get("USER", "lab_user")),
    description="ผู้แจ้ง:", continuous_update=False,
)
saved_refresh = widgets.Button(description="โหลดข้อมูลที่บันทึก", button_style="info")
saved_status = widgets.Label()
saved_tables = widgets.Accordion(children=[])


def refresh_saved_database(_=None):
    if globals().get("agent") is None:
        saved_tables.children = ()
        saved_status.value = "รันเซลล์ตั้งค่าและสร้าง Agent ก่อน (Mock ไม่บันทึกข้อมูล)"
        return
    user_ref = saved_user.value.strip()
    database = agent.store
    result = {}
    with database.transaction() as cursor:
        queries = {
            "users": "SELECT * FROM users WHERE user_ref=?",
            "cases": "SELECT * FROM cases WHERE user_ref=?",
            "case_raw_messages": "SELECT * FROM case_raw_messages WHERE user_ref=? ORDER BY created_at",
        }
        for table in ("case_personal_data", "case_drafts", "case_slots", "case_versions",
                      "case_version_contacts", "case_events", "pipeline_confirmed_cases", "pipeline_case_contacts"):
            queries[table] = f"SELECT * FROM {table} WHERE case_id IN (SELECT case_id FROM cases WHERE user_ref=?)"
        for table, sql in queries.items():
            rows = database._exec(cursor, sql, (user_ref,))
            columns = [column[0] for column in rows.description]
            result[table] = [dict(zip(columns, row)) for row in rows.fetchall()]
    panels = []
    for table, rows in result.items():
        panels.append(widgets.Textarea(
            value=json.dumps(rows, ensure_ascii=False, indent=2, default=str), disabled=True,
            layout=widgets.Layout(width="100%", height="260px"),
        ))
    saved_tables.children = tuple(panels)
    for index, (table, rows) in enumerate(result.items()):
        saved_tables.set_title(index, f"{table} — {len(rows)} แถว")
    saved_status.value = ("PostgreSQL" if database.postgres else "SQLite") + " · ข้อมูลจริงของ " + user_ref


saved_refresh.on_click(refresh_saved_database)
saved_user.observe(refresh_saved_database, names="value")
display(widgets.VBox([widgets.HBox([saved_user, saved_refresh]), saved_status, saved_tables]))
refresh_saved_database()
